# 10 — Indicateurs décisionnels

Objectif : produire une table d'indicateurs directement exploitables par un décideur, calculés
uniquement à partir des résultats réellement obtenus (notebooks 05 à 09).


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.data import load_processed_dataset
from src.analysis import (
    build_decision_indicators,
    gap_over_time,
    yearly_trend_by_group,
)
from src.modeling import (
    fit_mixed_effects,
    fit_ols,
    intraclass_correlation,
    prepare_model_frame,
    regression_metrics,
)
from src.utils.paths import TABLES_DIR, ensure_directory

df = load_processed_dataset("education_prioritaire")
print(df.shape)


(5250, 21)


## 1. Recalcul des briques nécessaires (tendances + métriques modèles)

In [2]:
trend_status = yearly_trend_by_group(df, value_col="variable_cible", group_col="statut", time_col="annee")
gap_repplus_horsep = gap_over_time(
    df,
    value_col="variable_cible",
    time_col="annee",
    group_col="statut",
    group_ref="Hors EP",
    group_compare="REP+",
)

explanatory = ["statut", "niveau", "ips", "dedoublement"]
categorical = ["statut", "niveau"]
frame = prepare_model_frame(df, "variable_cible", explanatory, categorical=categorical, group_column="ecole_id")

result_ols, _ = fit_ols(
    frame,
    target="variable_cible",
    explanatory=explanatory,
    categorical=categorical,
    cluster_column="ecole_id",
)
metrics = regression_metrics(frame["variable_cible"], result_ols.fittedvalues)

result_mixed, _ = fit_mixed_effects(
    frame,
    target="variable_cible",
    explanatory=explanatory,
    group_column="ecole_id",
    categorical=categorical,
)
icc = intraclass_correlation(result_mixed)

print(metrics)
print("ICC:", icc)


{'r2': 0.7030165920346809, 'rmse': 5.707194520578922, 'mae': 4.582196303051763, 'n': 5250}
ICC: 0.00678484092952671


## 2. Construction des indicateurs (5 à 10)

In [3]:
indicateurs = build_decision_indicators(
    df=df,
    trend_table=trend_status,
    gap_table=gap_repplus_horsep,
    model_metrics=metrics,
    icc=icc,
)
indicateurs


,nom,definition,formule,valeur,unite,population,interpretation,limites,utilisation_possible
0,Score global moyen,Moyenne de la variable cible sur l'ensemble de...,mean(variable_cible),78.231107,points sur 100,"Toutes observations (2017-2023, CP à 6e)",Niveau moyen de performance globale du système...,Agrégation globale masquant les écarts territo...,Repère macro pour suivre l'évolution générale ...
1,Écart REP+ vs Hors EP,Différence de score moyen entre REP+ et Hors EP.,mean(variable_cible|REP+) - mean(variable_cibl...,-10.249144,points sur 100,Statuts REP+ et Hors EP,Mesure brute de l'inégalité de performance ent...,"Indicateur associatif non causal, sensible à l...",Priorisation des dispositifs d'appui vers les ...
2,Tendance annuelle Hors EP,Pente de la tendance temporelle du score moyen...,slope(annee -> mean(variable_cible|Hors EP)),0.811284,points/an,Observations Hors EP,Variation moyenne annuelle du score Hors EP.,Tendance linéaire simplifiée sur 7 années.,Étalonnage d'une dynamique de référence hors EP.
3,Tendance annuelle REP+,Pente de la tendance temporelle du score moyen...,slope(annee -> mean(variable_cible|REP+)),0.836478,points/an,Observations REP+,Variation moyenne annuelle du score REP+.,Tendance linéaire simplifiée sur 7 années.,Suivi de l'intensité de progression en éducati...
4,Évolution de l'écart REP+ vs Hors EP (2017→2023),Variation de l'écart de score REP+ moins Hors ...,(gap_2023) - (gap_2017),-1.180823,points sur 100,REP+ et Hors EP par année,Négatif = creusement de l'écart en défaveur de...,Ne prouve pas un effet causal d'une politique.,Alerte stratégique sur réduction ou aggravatio...
5,Part d'observations dédoublées,Proportion d'observations avec dedoublement=1.,mean(dedoublement),0.335238,proportion (0-1),Toutes observations,Mesure l'intensité d'exposition au dédoublemen...,Ne renseigne pas à elle seule sur l'efficacité...,Suivi de couverture du dispositif.
6,RMSE modèle explicatif OLS,Erreur quadratique moyenne racine du modèle OL...,sqrt(mean((y - y_pred)^2)),5.707195,points sur 100,Toutes observations (in-sample),Ordre de grandeur de l'erreur moyenne de prédi...,"Mesure in-sample, non validée ici sur jeu exte...",Suivi de la robustesse d'un outil de pilotage ...
7,R² modèle explicatif OLS,Part de variance expliquée par le modèle OLS.,1 - SSE/SST,0.703017,proportion (0-1),Toutes observations (in-sample),Capacité explicative globale des variables ret...,Ne démontre pas la causalité des effets.,Comparer versions successives du modèle explic...
8,ICC effet école (modèle mixte),Part de variance résiduelle attribuable aux di...,var_inter_ecole / (var_inter_ecole + var_resid...,0.006785,proportion (0-1),Toutes observations,Faible valeur = faible surcroît d'information ...,Dépend de la spécification du modèle mixte.,Décider du niveau de granularité du pilotage (...


In [4]:
ensure_directory(TABLES_DIR)
out_path = TABLES_DIR / "indicateurs_decisionnels.csv"
indicateurs.to_csv(out_path, index=False)
print("Table enregistrée :", out_path)


Table enregistrée : C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\outputs\tables\indicateurs_decisionnels.csv


## 3. Lecture métier des indicateurs

Chaque indicateur fournit :
- un niveau global ou un écart d'inégalité ;
- une dynamique temporelle ;
- une métrique de performance du modèle de pilotage.

**Limite transversale** : ces indicateurs reposent sur des données simulées/reconstruites ; ils sont
méthodologiquement valides pour la chaîne analytique mais ne constituent pas des estimations officielles
de politique publique réelle sans recalage sur les données Open Data consolidées.